# Análisis de frecuencia de palabras

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

In [ ]:
file = "un-general-debates-blueprint.csv"

df = pd.read_csv("./dataset/"+file)
df.sample(2)

## Procesamiento

In [ ]:
import regex as re
import nltk

stopwords = set(nltk.corpus.stopwords.words('english'))

def tokenize(text):
    return re.findall(r'[\w-]*\p{L}[\w-]*',text)

def remove_stopword(tokens):
    return [t for t in tokens if t.lower() not in stopwords]

include_stopwords = {'dear', 'regards', 'must', 'would', 'also'}
exclude_stopwords = {'against'}

stopwords |= include_stopwords
stopwords -= exclude_stopwords

pipeline = [str.lower, tokenize, remove_stopword]

def prepare(text, pipeline):
    tokens = text
    for transform in pipeline:
        tokens = transform(tokens)
    return tokens

In [ ]:
df['tokens'] = df['text'].apply(prepare, pipeline= pipeline)
df['num_tokens'] = df['tokens'].map(len)

## Contador de  palabras con `Counter`

In [ ]:
def count_words(df, column='tokens', preprocess=None, min_freq=2):
    # process tokens and update counter
    def update(doc):
        tokens = doc if preprocess is None else preprocess(doc)
        counter.update(tokens)

    # create counter and run through all data
    counter = Counter()
    df[column].map(update)

    # transform counter into a DataFrame
    freq_df = pd.DataFrame.from_dict(counter, orient='index', columns=['freq'])
    freq_df = freq_df.query('freq >= @min_freq')
    freq_df.index.name = 'token'

    return freq_df.sort_values('freq', ascending=False)    

Para eficientar el proceso de realizar el conteo sobre un corpus, se utiliza `map` del DataFrame. 

Para un posterior preprocesamiento es preferible transformar el counter en un DataFrame. Los tokens son los índices del DataFrame, mientras los valores de frecuencia son almacenados en una columna llamada `freq`. Los registros son ordenados de manera que la palabra más frecuente aparece al inicio.

In [ ]:
freq_df = count_words(df)
freq_df.head(5)

Si no se desea utilizar tokens precalculados para algunos análisis especiales, se pede tokenizar el texto en el aire con una función de preprocesamiento como un tercer parámetro. Por ejemplo, se puede generar y contar todas las palabras con diez o más carácteres con la tokenización al aire para el texto:

In [ ]:
count_words(df, column='text', 
            preprocess=lambda text: re.findall(r"\w{10,}", text))

## Representación visual de la frecuencia de palabras

In [ ]:
ax = freq_df.head(15).plot(kind='barh', width=0.95)
ax.invert_yaxis()
ax.set(xlabel='Frequency', ylabel='Token', title='Top Words')

## Nube de palabras (wordcloud)

In [ ]:
from wordcloud import WordCloud

In [ ]:
text = df.query("year==2015 and country=='USA'")['text'].values[0]

wc = WordCloud(max_words=100, stopwords=stopwords)
wc.generate(text)
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')

Este código sirve para un solo texto.  En este caso se construye una función que soporte Pandas Series que contiene los valores de frecuencia. 

In [ ]:
def wordcloud(word_freq, title=None, max_words=200, stopwords=None):
    wc = WordCloud(width=800, height=400,
                   background_color='black', colormap='Paired',
                   max_font_size=150, max_words=max_words)
    
    # covert DataFrame into dict
    if type(word_freq) == pd.Series:
        counter = Counter(word_freq.fillna(0).to_dict())
    else:
        counter = word_freq

    # filter stop words in frequency counter
    if stopwords is not None:
        counter = {token:freq for (token, freq) in counter.items() if token not in stopwords}

    wc.generate_from_frequencies(counter)
    plt.title(title)
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')

In [ ]:
freq_2015_df = count_words(df[df['year']==2015])
plt.figure()
wordcloud(freq_2015_df['freq'], max_words=100)

En el segundo filtro se agrega una lista de stopword. En ocasiones es util filtrar frecuencias específicas pero no interesantes para la visualización solamente.

In [ ]:
wordcloud(freq_2015_df['freq'], max_words=100, stopwords=freq_df.head(50).index)

Claramente el segundo wc da una idea mucho mejor de los temas del 2015, pero aun son frecuentes palabras como today y challenges. Es necesario dar un menor peso a estas palabras. como se muestra a continuación.

# Ejercicio

A partir del UN General Debate Dataset, genera los histogramas de frecuencia y las nubes de palabras correspondientes a los años 1970, 1980, 1990, 2000 y 2010.

Consideraciones:
	•	Elimina las palabras de alta frecuencia (stopwords) para obtener resultados más representativos.